# S4 · AndinaLog 03B · Notebook 2 · Tratamiento controlado de telemetría IoT

Este notebook trabaja únicamente con `andinalog_iot_telemetry.csv`. Lee las salidas del notebook 1 y el `catalogo_reglas_tratamiento.csv` de esta carpeta. Solo aplica una regla cuando el catálogo indica `APROBADA`. Conserva todas las filas originales y documenta los tratamientos aplicados y los problemas que permanecen pendientes. No fuerza ninguna fila a salir de cuarentena.

El catálogo es una decisión del proyecto, no una inferencia automática. Antes de cambiar una regla de `PENDIENTE` a `APROBADA`, documenta su evidencia y validación. El informe final se genera a partir de lo que **realmente ocurrió** en la ejecución.


## 1 · Configuración y entradas

En local, ejecuta desde cualquier carpeta del proyecto. En Colab, selecciona `drive` y ajusta la ruta de la carpeta que contiene `datasets/` y `S4/`. El notebook 2 lee los dos CSV del notebook 1 y guarda las salidas en `S4/andinalog_iot_telemetry/notebook2/salidas/`.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd
import numpy as np

ENTORNO = "local"  # "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"  # ajustar si corresponde
VERSION_TRATAMIENTO = "GIAD-M3-S4-IOT-tratamiento-v1"

def raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / "AndinaLog_03B_Bronce").is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("Ejecuta dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = raiz_local()
    else:
        raise ValueError("ENTORNO debe ser local o drive")
    caso = raiz / "S4" / "andinalog_iot_telemetry"
    return {
        "bronze": raiz / "datasets" / "AndinaLog_03B_Bronce" / "andinalog_iot_telemetry.csv",
        "principal": caso / "notebook1" / "salidas" / "andinalog_iot_telemetry_diagnosticado.csv",
        "problemas": caso / "notebook1" / "salidas" / "andinalog_iot_telemetry_problemas.csv",
        "reporte1": caso / "notebook1" / "salidas" / "andinalog_iot_telemetry_reporte_calidad.csv",
        "catalogo": caso / "notebook2" / "catalogo_reglas_tratamiento.csv",
        "salidas": caso / "notebook2" / "salidas",
    }

rutas = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
for nombre in ["bronze", "principal", "problemas", "reporte1", "catalogo"]:
    if not rutas[nombre].is_file():
        raise FileNotFoundError(f"Falta {nombre}: {rutas[nombre]}")
print("Entradas verificadas; salidas:", rutas["salidas"])


## 2 · Lectura y validación del contrato

El número `fila_bronze` vincula el archivo principal con el detalle de problemas. La huella del reporte 1 debe corresponder al CSV Bronze actual; si el origen cambió, se debe volver a ejecutar el notebook 1 antes de curar.


In [ ]:
def leer_entradas(rutas):
    principal = pd.read_csv(rutas["principal"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    problemas = pd.read_csv(rutas["problemas"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    reporte1 = pd.read_csv(rutas["reporte1"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    catalogo = pd.read_csv(rutas["catalogo"], dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return principal, problemas, reporte1, catalogo

def validar_entradas(principal, problemas, reporte1, catalogo, ruta_bronze):
    requeridas = {"fila_bronze", "timestamp", "viaje_id", "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct", "en_cuarentena"}
    if not requeridas.issubset(principal.columns):
        raise ValueError(f"Faltan columnas del principal: {sorted(requeridas - set(principal.columns))}")
    if not {"fila_bronze", "columna_afectada", "codigo_error", "valor_original"}.issubset(problemas.columns):
        raise ValueError("El detalle de problemas no cumple su contrato")
    if not {"regla_id", "estado", "evidencia_acuerdo"}.issubset(catalogo.columns):
        raise ValueError("El catálogo no cumple su contrato")
    if principal["fila_bronze"].duplicated().any() or catalogo["regla_id"].duplicated().any():
        raise ValueError("Hay identificadores duplicados")
    if not problemas["fila_bronze"].isin(principal["fila_bronze"]).all():
        raise ValueError("Hay problemas sin fila en el principal")
    if not catalogo["estado"].isin(["APROBADA", "PENDIENTE"]).all():
        raise ValueError("Estado de regla desconocido")
    hash_reporte = reporte1.set_index("metrica").loc["sha256_bronze", "valor"]
    hash_actual = hashlib.sha256(ruta_bronze.read_bytes()).hexdigest()
    if hash_reporte != hash_actual:
        raise ValueError("El Bronze ya no coincide con el diagnóstico; ejecuta el notebook 1")
    if len(principal) != int(reporte1.set_index("metrica").loc["filas_bronze", "valor"]):
        raise ValueError("El principal no coincide con el conteo Bronze")
    return hash_actual

df_entrada, problemas_entrada, reporte1, catalogo = leer_entradas(rutas)
HASH_BRONZE = validar_entradas(df_entrada, problemas_entrada, reporte1, catalogo, rutas["bronze"])
print(f"Filas: {len(df_entrada):,}; problemas iniciales: {len(problemas_entrada):,}")
display(catalogo)


## 3 · Aplicación de reglas aprobadas

El valor original no se sobrescribe. La conversión C/F se guarda en `temperatura_c_preparada` cuando su regla está aprobada. Kelvin se convierte solo si la aprobación confirma explícitamente que `K` es kelvin. Fechas imposibles, humedades negativas y faltantes no se corrigen por conjetura. Una copia idéntica puede quedar identificada como excluida de la vista utilizable, pero permanece en el archivo completo y en cuarentena como copia.


In [ ]:
def regla_aprobada(catalogo, regla_id):
    fila = catalogo.loc[catalogo["regla_id"].eq(regla_id)]
    if len(fila) != 1:
        raise ValueError(f"Falta regla única: {regla_id}")
    aprobada = fila.iloc[0]["estado"] == "APROBADA"
    if aprobada and not str(fila.iloc[0]["evidencia_acuerdo"]).strip():
        raise ValueError(f"La regla {regla_id} figura aprobada sin evidencia del acuerdo")
    return aprobada

def preparar_temperatura(principal, catalogo):
    df = principal.copy(deep=True)
    numero = pd.to_numeric(df["temperatura_cabina_c"].str.strip(), errors="coerce")
    unidad = df["temp_unit"].str.strip().str.upper()
    df["temperatura_c_preparada"] = np.nan
    df["tratamiento_temperatura"] = "SIN_TRATAMIENTO"
    if regla_aprobada(catalogo, "TEMP_C_CONSERVAR"):
        c = unidad.eq("C") & numero.notna()
        df.loc[c, "temperatura_c_preparada"] = numero[c]
        df.loc[c, "tratamiento_temperatura"] = "C_ORIGINAL"
    if regla_aprobada(catalogo, "TEMP_F_A_C"):
        f = unidad.eq("F") & numero.notna()
        df.loc[f, "temperatura_c_preparada"] = (numero[f] - 32) * 5 / 9
        df.loc[f, "tratamiento_temperatura"] = "F_A_C"
    if regla_aprobada(catalogo, "TEMP_K_A_C"):
        k = unidad.eq("K") & numero.notna() & numero.ge(0)
        df.loc[k, "temperatura_c_preparada"] = numero[k] - 273.15
        df.loc[k, "tratamiento_temperatura"] = "K_A_C"
    df["temperatura_c_preparada"] = df["temperatura_c_preparada"].round(6)
    return df

def evaluar_duplicado_identico(principal, problemas):
    raw = ["timestamp", "viaje_id", "order_id", "camion_id", "producto_id", "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct", "desviacion_termica_flag", "desviacion_proximos_60min_flag"]
    clave = principal["viaje_id"].str.strip() + "|" + principal["timestamp"].str.strip()
    canonica = principal.loc[~clave.duplicated(keep="first")].set_index(clave[~clave.duplicated(keep="first")])
    copias = problemas.loc[problemas["codigo_error"].eq("DUPLICADO"), "fila_bronze"]
    resultado = {}
    for fila in copias:
        registro = principal.loc[principal["fila_bronze"].eq(fila)].iloc[0]
        original = canonica.loc[registro["viaje_id"].strip() + "|" + registro["timestamp"].strip()]
        resultado[fila] = all(registro[col] == original[col] for col in raw)
    return resultado

def evaluar_problemas(principal, problemas, catalogo):
    acciones = problemas.copy(deep=True)
    acciones["estado_tratamiento"] = "PENDIENTE"
    acciones["tratamiento_aplicado"] = "NINGUNO"
    acciones["detalle_resultado"] = "No hay regla aprobada y validada para liberar esta fila"
    if regla_aprobada(catalogo, "TEMP_K_A_C"):
        k = acciones["columna_afectada"].eq("temp_unit") & acciones["codigo_error"].eq("UNIDAD_NO_RECONOCIDA") & acciones["valor_original"].str.strip().str.upper().eq("K")
        temperatura = principal.set_index("fila_bronze")["temperatura_c_preparada"]
        resuelta = k & acciones["fila_bronze"].map(temperatura).notna()
        acciones.loc[resuelta, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = ["RESUELTO", "K_A_C", "Unidad confirmada y conversión validada; revisar otros problemas de la fila"]
    if regla_aprobada(catalogo, "DUPLICADO_IDENTICO"):
        iguales = evaluar_duplicado_identico(principal, acciones)
        dup = acciones["codigo_error"].eq("DUPLICADO") & acciones["fila_bronze"].map(iguales).fillna(False)
        acciones.loc[dup, ["estado_tratamiento", "tratamiento_aplicado", "detalle_resultado"]] = ["EXCLUIDO_COMO_COPIA", "COPIA_IDENTICA", "Copia idéntica conservada para auditoría; no es una lectura nueva utilizable"]
    return acciones

df_trabajo = preparar_temperatura(df_entrada, catalogo)
acciones = evaluar_problemas(df_trabajo, problemas_entrada, catalogo)
print(acciones["estado_tratamiento"].value_counts().to_string())


## 4 · Estado final y conciliación

Solo `RESUELTO` deja de ser motivo pendiente. `EXCLUIDO_COMO_COPIA` conserva la fila en cuarentena para impedir que duplique una lectura, aunque su tratamiento queda registrado. Una fila sale de cuarentena únicamente cuando no le queda ningún problema pendiente o excluido.


In [ ]:
def construir_estado_final(principal, acciones):
    df = principal.copy(deep=True)
    df["en_cuarentena_inicial"] = df["en_cuarentena"].str.lower().eq("true")
    df["columnas_problema_iniciales"] = df["columnas_con_problemas"]
    pendientes = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].copy()
    pendientes["motivo"] = pendientes["columna_afectada"] + ":" + pendientes["codigo_error"]
    motivo_por_fila = pendientes.groupby("fila_bronze")["motivo"].agg(lambda v: "|".join(v))
    df["motivos_finales"] = df["fila_bronze"].map(motivo_por_fila).fillna("")
    df["en_cuarentena_final"] = df["motivos_finales"].ne("")
    tratamientos = acciones.loc[acciones["tratamiento_aplicado"].ne("NINGUNO")].groupby("fila_bronze")["tratamiento_aplicado"].agg(lambda v: "|".join(dict.fromkeys(v)))
    tratamiento_medicion = df["tratamiento_temperatura"].where(df["tratamiento_temperatura"].isin(["F_A_C", "K_A_C"]), "")
    df["tratamientos_aplicados"] = ["|".join(dict.fromkeys([x for x in [t, a] if x]))
                                   for t, a in zip(tratamiento_medicion, df["fila_bronze"].map(tratamientos).fillna(""))]
    df["version_tratamiento"] = VERSION_TRATAMIENTO
    return df, pendientes

df_final, problemas_pendientes = construir_estado_final(df_trabajo, acciones)
df_cuarentena_final = df_final.loc[df_final["en_cuarentena_final"]].copy()
pd.testing.assert_frame_equal(df_final[df_entrada.columns], df_entrada)
assert len(df_final) == len(df_entrada)
assert len(df_cuarentena_final) == int(df_final["en_cuarentena_final"].sum())
assert set(problemas_pendientes["fila_bronze"]) == set(df_cuarentena_final["fila_bronze"])
assert not (df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).any() or (acciones["estado_tratamiento"].eq("RESUELTO")).any()
print("Filas iniciales en cuarentena:", int(df_final["en_cuarentena_inicial"].sum()))
print("Filas finales en cuarentena:", int(df_final["en_cuarentena_final"].sum()))
print("Filas liberadas:", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum()))


## 5 · Informe de lo aplicado y exportación

El reporte registra las reglas aprobadas, los problemas resueltos, los pendientes, las copias excluidas y el número real de filas liberadas. No atribuye una liberación a una conversión si la fila mantiene otro motivo.


In [ ]:
def crear_reporte(df_final, acciones, catalogo, huella):
    base = [
        ("sha256_bronze", huella),
        ("version_tratamiento", VERSION_TRATAMIENTO),
        ("filas_totales", len(df_final)),
        ("filas_cuarentena_inicial", int(df_final["en_cuarentena_inicial"].sum())),
        ("filas_cuarentena_final", int(df_final["en_cuarentena_final"].sum())),
        ("filas_liberadas", int((df_final["en_cuarentena_inicial"] & ~df_final["en_cuarentena_final"]).sum())),
        ("problemas_resueltos", int(acciones["estado_tratamiento"].eq("RESUELTO").sum())),
        ("copias_excluidas", int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())),
        ("problemas_pendientes", int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())),
        ("temperaturas_C_conservadas", int(df_final["tratamiento_temperatura"].eq("C_ORIGINAL").sum())),
        ("temperaturas_F_a_C", int(df_final["tratamiento_temperatura"].eq("F_A_C").sum())),
        ("temperaturas_K_a_C", int(df_final["tratamiento_temperatura"].eq("K_A_C").sum())),
    ]
    base += [("regla_" + r["regla_id"], r["estado"]) for _, r in catalogo.iterrows()]
    return pd.DataFrame(base, columns=["metrica", "valor"])

def exportar(directorio, tablas, bronze, huella):
    if hashlib.sha256(bronze.read_bytes()).hexdigest() != huella:
        raise RuntimeError("El Bronze cambió; se cancela la exportación")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_iot2_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

reporte2 = crear_reporte(df_final, acciones, catalogo, HASH_BRONZE)
tablas = {
    "andinalog_iot_telemetry_tratado.csv": df_final,
    "andinalog_iot_telemetry_acciones.csv": acciones,
    "andinalog_iot_telemetry_cuarentena_final.csv": df_cuarentena_final,
    "andinalog_iot_telemetry_reporte_tratamiento.csv": reporte2,
}
for ruta in exportar(rutas["salidas"], tablas, rutas["bronze"], HASH_BRONZE):
    print(ruta)
display(reporte2)


In [ ]:
def crear_informe_md(df_final, acciones, catalogo, huella):
    inicial = int(df_final["en_cuarentena_inicial"].sum())
    final = int(df_final["en_cuarentena_final"].sum())
    liberadas = inicial - final
    f = int(df_final["tratamiento_temperatura"].eq("F_A_C").sum())
    k = int(df_final["tratamiento_temperatura"].eq("K_A_C").sum())
    resueltos = int(acciones["estado_tratamiento"].eq("RESUELTO").sum())
    pendientes = int(acciones["estado_tratamiento"].eq("PENDIENTE").sum())
    excluidos = int(acciones["estado_tratamiento"].eq("EXCLUIDO_COMO_COPIA").sum())
    conteos = acciones.loc[~acciones["estado_tratamiento"].eq("RESUELTO")].groupby(["columna_afectada", "codigo_error"]).size()
    lineas = [
        "# Informe de tratamiento de telemetría IoT",
        "",
        f"**Fuente Bronze SHA-256:** `{huella}`",
        f"**Versión:** `{VERSION_TRATAMIENTO}`",
        "",
        "## Resultado del lote",
        "",
        f"Se conservaron las {len(df_final):,} filas. La cuarentena pasó de {inicial:,} a {final:,} filas; {liberadas:,} salieron después de resolver todos sus motivos. Se registraron {resueltos:,} problemas resueltos, {pendientes:,} pendientes y {excluidos:,} copias excluidas de la vista utilizable.",
        "",
        "## Tratamientos aplicados",
        "",
        f"- Se convirtieron {f:,} temperaturas de Fahrenheit a Celsius con `(F − 32) × 5/9`, conservando el valor original. Estas lecturas no estaban en cuarentena únicamente por la unidad.",
        f"- Se convirtieron {k:,} temperaturas de kelvin a Celsius con `K − 273,15`, según la confirmación del usuario. Cuatro filas salieron de cuarentena; la quinta conserva un motivo adicional de humedad faltante.",
        "- No se imputaron temperaturas ni humedades. No se corrigieron fechas imposibles ni humedades negativas por suposición.",
        "- La regla para copias duplicadas continúa pendiente de acuerdo; no se liberó ni descartó ninguna por ese motivo.",
        "",
        "## Problemas que permanecen",
        "",
        "| Columna | Código | Hallazgos pendientes |",
        "|---|---|---:|",
    ]
    lineas += [f"| {col} | {codigo} | {int(total)} |" for (col, codigo), total in conteos.items()]
    lineas += [
        "",
        "La cantidad de hallazgos pendientes puede superar las filas en cuarentena porque una fila puede presentar más de un problema. Cada intento y su resultado figuran en el CSV de acciones; el CSV tratado mantiene los valores originales y el estado inicial y final.",
        "",
        "## Reglas y límites de esta ejecución",
        "",
    ]
    lineas += [f"- `{r['regla_id']}`: **{r['estado']}**. {r['evidencia_acuerdo'] or 'Sin acuerdo documentado.'}"
               for _, r in catalogo.iterrows()]
    lineas += ["", "Una fila permaneció en cuarentena cuando no había una regla aprobada, faltaba evidencia o quedaba otro motivo sin resolver. No se forzó ninguna liberación.", ""]
    return "\n".join(lineas)

informe_md = crear_informe_md(df_final, acciones, catalogo, HASH_BRONZE)
ruta_informe = rutas["salidas"] / "Informe_S4_02_Tratamiento.md"
ruta_informe.write_text(informe_md, encoding="utf-8")
print(ruta_informe)


## 6 · Interpretación de esta ejecución

Revisa el reporte y el archivo de acciones antes de afirmar que una fila salió de cuarentena. Las reglas que siguen `PENDIENTE` requieren evidencia o acuerdo; por eso no se aplica una imputación, una fecha supuesta ni un valor de humedad fabricado. El informe de curación debe describir exactamente los intentos ejecutados, sus resultados y los casos que permanecen en cuarentena.
